In [4]:
import os
import re

folder = "outputs/plaintext"

for filename in os.listdir(folder):
    if not filename.endswith(".txt"):
        continue

    # 匹配形如 xxx_38_en.txt
    new_name = re.sub(
        r'_(\d+)_en\.txt$',
        lambda m: f"_{int(m.group(1)):03d}_en.txt",
        filename
    )

    if new_name != filename:
        src = os.path.join(folder, filename)
        dst = os.path.join(folder, new_name)
        os.rename(src, dst)
        print(f"✅ {filename} -> {new_name}")

✅ Jer_26_en.txt -> Jer_026_en.txt
✅ Mk_5_en.txt -> Mk_005_en.txt
✅ Rom_8_en.txt -> Rom_008_en.txt
✅ Rev_20_en.txt -> Rev_020_en.txt
✅ Ezk_46_en.txt -> Ezk_046_en.txt
✅ Am_7_en.txt -> Am_007_en.txt
✅ 2S_4_en.txt -> 2S_004_en.txt
✅ Jer_38_en.txt -> Jer_038_en.txt
✅ Lev_9_en.txt -> Lev_009_en.txt
✅ 2K_2_en.txt -> 2K_002_en.txt
✅ 2Chr_9_en.txt -> 2Chr_009_en.txt
✅ Pro_4_en.txt -> Pro_004_en.txt
✅ Zec_9_en.txt -> Zec_009_en.txt
✅ Ezk_25_en.txt -> Ezk_025_en.txt
✅ Wis_12_en.txt -> Wis_012_en.txt
✅ Is_14_en.txt -> Is_014_en.txt
✅ Acts_15_en.txt -> Acts_015_en.txt
✅ Jer_45_en.txt -> Jer_045_en.txt
✅ Num_10_en.txt -> Num_010_en.txt
✅ Sir_31_en.txt -> Sir_031_en.txt
✅ Sir_40_en.txt -> Sir_040_en.txt
✅ 2Chr_5_en.txt -> 2Chr_005_en.txt
✅ Es_06_Chapter_4_en.txt -> Es_06_Chapter_004_en.txt
✅ Is_65_en.txt -> Is_065_en.txt
✅ Lev_5_en.txt -> Lev_005_en.txt
✅ Jer_34_en.txt -> Jer_034_en.txt
✅ 2S_8_en.txt -> 2S_008_en.txt
✅ 2S_16_en.txt -> 2S_016_en.txt
✅ Is_18_en.txt -> Is_018_en.txt
✅ 1K_13_en.txt -> 1

In [1]:
import sqlite3

db = sqlite3.connect("db/bible.db")
cur = db.cursor()

# 执行替换
cur.execute("""
    UPDATE tokens
    SET align_id = REPLACE(align_id, '.', '_')
    WHERE align_id LIKE '%.%'
""")

db.commit()
print("受影响的行数:", cur.rowcount)

db.close()

受影响的行数: 4537


In [9]:
import sqlite3

db_path = "db/bible.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# 批量更新：以 Mt 开头的 id，前面加上 47_
sql = """
UPDATE timestamps
SET id = '01_' || id
WHERE id LIKE 'Gen%'
"""

cursor.execute(sql)
conn.commit()

print(f"共更新了 {cursor.rowcount} 条记录")

conn.close()

共更新了 778 条记录


In [10]:
import sqlite3

db_path = "db/bible.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# 1️⃣ 查询所有需要更新的 id
cursor.execute('SELECT id FROM timestamps')
rows = cursor.fetchall()

update_sql = 'UPDATE timestamps SET id = ? WHERE id = ?'
updates = []

for row in rows:
    old_id = row[0]
    
    if '__' not in old_id:
        continue  # 防止格式不符合的数据
    
    prefix, num = old_id.split('__', 1)
    
    if not num.isdigit():
        continue  # 防止非数字情况
    
    new_id = f"{prefix}__{num.zfill(4)}"
    
    if new_id != old_id:
        updates.append((new_id, old_id))

# 2️⃣ 批量更新
cursor.executemany(update_sql, updates)
conn.commit()

print(f"✅ 共更新 {len(updates)} 条记录")

conn.close()

✅ 共更新 4299 条记录


In [5]:
import sqlite3

db_path = "db/bible.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# 批量更新 tokens 表：align_id 以 Mt 开头的，前面加 47_
sql = """
UPDATE tokens
SET align_id = '52_' || align_id
WHERE align_id LIKE 'Rom%'
"""

cursor.execute(sql)
conn.commit()

print(f"tokens 表中共更新了 {cursor.rowcount} 条记录")

conn.close()

tokens 表中共更新了 509 条记录


In [6]:
import sqlite3
import re

db_path = "db/bible.db"

# 连接数据库
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# 先查看一下当前数据
cursor.execute("SELECT id FROM timestamps LIMIT 10")
print("修改前：", cursor.fetchall())

def transform_id(old_id):
    """
    将形如 47_Mt_1__1 转换为 47_Mt_001__0001
    - 第一个数字（下划线之前）补0到3位
    - 第二个数字（下划线之后）补0到4位
    """
    # 用正则匹配：数字_文字_数字__数字 的模式
    pattern = r'^(\d+)_([A-Za-z]+)_(\d+)__(\d+)$'
    m = re.match(pattern, old_id)
    if m:
        num1 = m.group(1).zfill(3)   # 补0到3位
        text = m.group(2)            # 中间文字不变
        num2 = m.group(3).zfill(4)   # 补0到4位
        num3 = m.group(4).zfill(4)   # 补0到4位
        return f"{num1}_{text}_{num2}__{num3}"
    return old_id  # 不匹配的保持原样

# 取出所有 id
cursor.execute("SELECT id FROM timestamps")
rows = cursor.fetchall()

# 批量更新
update_sql = "UPDATE timestamps SET id = ? WHERE id = ?"
updates = []
for row in rows:
    old_id = row[0]
    new_id = transform_id(old_id)
    if old_id != new_id:
        updates.append((new_id, old_id))

# 执行更新
cursor.executemany(update_sql, updates)
conn.commit()

print(f"共更新了 {len(updates)} 条记录")

# 验证结果
cursor.execute("SELECT id FROM timestamps LIMIT 10")
print("修改后：", cursor.fetchall())

conn.close()

修改前： [('01_Gen_1__1',), ('01_Gen_1__10',), ('01_Gen_1__100',), ('01_Gen_1__101',), ('01_Gen_1__102',), ('01_Gen_1__103',), ('01_Gen_1__104',), ('01_Gen_1__105',), ('01_Gen_1__106',), ('01_Gen_1__107',)]
共更新了 3300 条记录
修改后： [('001_Gen_0001__0001',), ('001_Gen_0001__0002',), ('001_Gen_0001__0003',), ('001_Gen_0001__0004',), ('001_Gen_0001__0005',), ('001_Gen_0001__0006',), ('001_Gen_0001__0007',), ('001_Gen_0001__0008',), ('001_Gen_0001__0009',), ('001_Gen_0001__0010',)]


In [9]:
import sqlite3
import re
import shutil
from datetime import datetime

# ========== 配置 ==========
db_path = "db/bible.db"
backup_path = f"db/bible.db.bak.{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# ========== 1. 备份数据库 ==========
shutil.copy(db_path, backup_path)
print(f"✅ 已备份数据库：{backup_path}")

# ========== 2. 连接数据库 ==========
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = OFF")

# ========== 3. 转换函数 ==========
def transform_id(old_id: str) -> str:
    """
    形如：
    TEXT_NUMBERA__NUMBERB
    示例：
    47_Mt_1__1 → 47_Mt_001__0001
    """
    m = re.match(r'^(.+?)_(\d+)(__)(\d+)$', old_id)
    if not m:
        return old_id
    prefix = m.group(1)   # 如 47_Mt
    a = m.group(2).zfill(3)
    sep = m.group(3)
    b = m.group(4).zfill(4)
    return f"{prefix}_{a}{sep}{b}"

# ========== 4. 查看原始数据 ==========
cursor.execute("SELECT id FROM timestamps")
rows = cursor.fetchall()

print("\n📌 原始数据示例：")
for r in rows[:10]:
    print("  ", r[0])

# ========== 5. 批量更新 ==========
updated = 0
for (old_id,) in rows:
    new_id = transform_id(old_id)
    if new_id != old_id:
        cursor.execute(
            "UPDATE timestamps SET id = ? WHERE id = ?",
            (new_id, old_id)
        )
        updated += 1

conn.commit()
print(f"\n✅ 共更新 {updated} 条记录")

# ========== 6. 恢复外键 ==========
cursor.execute("PRAGMA foreign_keys = ON")

# ========== 7. 校验结果 ==========
cursor.execute("SELECT id FROM timestamps ORDER BY id LIMIT 10")
print("\n📌 更新后数据示例：")
for r in cursor.fetchall():
    print("  ", r[0])

conn.close()
print("\n🎉 全部完成")

✅ 已备份数据库：db/bible.db.bak.20260701_232312

📌 原始数据示例：
   01_Gen_1__1
   01_Gen_1__10
   01_Gen_1__100
   01_Gen_1__101
   01_Gen_1__102
   01_Gen_1__103
   01_Gen_1__104
   01_Gen_1__105
   01_Gen_1__106
   01_Gen_1__107

✅ 共更新 4534 条记录

📌 更新后数据示例：
   01_Gen_001__0001
   01_Gen_001__0002
   01_Gen_001__0003
   01_Gen_001__0004
   01_Gen_001__0005
   01_Gen_001__0006
   01_Gen_001__0007
   01_Gen_001__0008
   01_Gen_001__0009
   01_Gen_001__0010

🎉 全部完成


In [8]:
import os
import shutil
import glob

db_path = "db/bible.db"

# 找到最新的备份文件（按时间戳排序）
backup_files = sorted(
    glob.glob("db/bible.db.bak.*"),
    key=os.path.getmtime,
    reverse=True
)

if not backup_files:
    print("❌ 没有找到任何 bible.db.bak.* 备份文件")
else:
    latest_backup = backup_files[0]
    
    # 恢复
    shutil.copy(latest_backup, db_path)
    print(f"✅ 已从备份恢复：{latest_backup}")
    print(f"📂 当前数据库：{db_path}")

✅ 已从备份恢复：db/bible.db.bak.20260701_232103
📂 当前数据库：db/bible.db


In [10]:
import sqlite3
import re
import shutil
from datetime import datetime

# ========== 配置 ==========
db_path = "db/bible.db"
backup_path = f"db/bible.db.bak.{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# ========== 1. 备份数据库 ==========
shutil.copy(db_path, backup_path)
print(f"✅ 已备份数据库：{backup_path}")

# ========== 2. 连接数据库 ==========
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# 关闭外键约束（避免 UPDATE 报错）
cursor.execute("PRAGMA foreign_keys = OFF")

# ========== 3. 转换函数 ==========
def transform_id(old_id: str) -> str:
    """
    通用规则：
    TEXT_NUMBERA__NUMBERB
    示例：
    47_Mt_1__1 → 47_Mt_001__0001
    """
    if not old_id:
        return old_id

    m = re.match(r'^(.+?)_(\d+)(__)(\d+)$', old_id)
    if not m:
        return old_id

    prefix = m.group(1)
    a = m.group(2).zfill(3)
    sep = m.group(3)
    b = m.group(4).zfill(4)

    return f"{prefix}_{a}{sep}{b}"

# ========== 4. 更新 timestamps.id ==========
print("\n🔄 更新 timestamps.id ...")

cursor.execute("SELECT id FROM timestamps")
rows = cursor.fetchall()

updated_ts = 0
for (old_id,) in rows:
    new_id = transform_id(old_id)
    if new_id != old_id:
        cursor.execute(
            "UPDATE timestamps SET id = ? WHERE id = ?",
            (new_id, old_id)
        )
        updated_ts += 1

print(f"✅ timestamps 更新 {updated_ts} 条")

# ========== 5. 更新 tokens.align_id ==========
print("\n🔄 更新 tokens.align_id ...")

cursor.execute("SELECT align_id FROM tokens")
rows = cursor.fetchall()

updated_tk = 0
for (old_id,) in rows:
    new_id = transform_id(old_id)
    if new_id != old_id:
        cursor.execute(
            "UPDATE tokens SET align_id = ? WHERE align_id = ?",
            (new_id, old_id)
        )
        updated_tk += 1

print(f"✅ tokens 更新 {updated_tk} 条")

# ========== 6. 提交 & 恢复外键 ==========
conn.commit()
cursor.execute("PRAGMA foreign_keys = ON")

# ========== 7. 校验结果 ==========
print("\n📌 timestamps 示例：")
cursor.execute("SELECT id FROM timestamps ORDER BY id LIMIT 5")
for r in cursor.fetchall():
    print(" ", r[0])

print("\n📌 tokens.align_id 示例：")
cursor.execute("SELECT align_id FROM tokens ORDER BY align_id LIMIT 5")
for r in cursor.fetchall():
    print(" ", r[0])

conn.close()
print("\n🎉 全部完成")

✅ 已备份数据库：db/bible.db.bak.20260701_232422

🔄 更新 timestamps.id ...
✅ timestamps 更新 0 条

🔄 更新 tokens.align_id ...
✅ tokens 更新 4537 条

📌 timestamps 示例：
  01_Gen_001__0001
  01_Gen_001__0002
  01_Gen_001__0003
  01_Gen_001__0004
  01_Gen_001__0005

📌 tokens.align_id 示例：
  None
  None
  None
  None
  None

🎉 全部完成
